# Notebook 09 orchestration

Tagged definitions only. Run All in the main consumer launches the actual pilot in detached tmux. All four architectures and two new seed pairs train for exactly 350 updates if calibration confirms feasibility. The original three-hour clock includes preflight, training and all 16 evaluations. No seed, update budget or endpoint is changed after seeing outcomes. A cutoff is INCOMPLETE, never a winner. This module does not launch the experiment when opened.

In [ ]:
"""Notebook09 orchestration: one persisted three-hour experimental clock."""
from pathlib import Path
import copy, hashlib, importlib.util, json, os, shlex, subprocess, sys, time
from datetime import datetime, timezone
import nbformat
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json, digest_file, run_writer
MAIN_NOTEBOOK='notebooks/09_Deep_Energy_Encoder_Pilot.ipynb'
TERMINAL={'COMPLETED','FAILED','INCOMPLETE'}

def _read(path):return json.loads(Path(path).read_text())
def _normalized(value):return json.loads(json.dumps(value,allow_nan=False))
def _hash(value):return hashlib.sha256(json.dumps(_normalized(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def _export_hash(path):
    return hashlib.sha256('\n'.join(''.join(c['source']) for c in _read(path)['cells'] if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])).encode()).hexdigest()
def _load(root,name):
    key='fmm09_'+name
    if key not in sys.modules:
        spec=importlib.util.spec_from_loader(key,TaggedNotebookLoader(Path(root)/'experiments/09_deep_energy'/(name+'.ipynb')))
        module=importlib.util.module_from_spec(spec);sys.modules[key]=module
        try:spec.loader.exec_module(module)
        except BaseException:sys.modules.pop(key,None);raise
    return sys.modules[key]

def source_hashes(root):
    return {str(p.relative_to(root)):_export_hash(p) for folder in ('fmm_jepa','experiments/04_followup','experiments/05_scaling','experiments/07_energy','experiments/08_replication','experiments/09_deep_energy') for p in sorted((Path(root)/folder).glob('*.ipynb'))}

def consumer_hash(root):return _hash([''.join(c['source']) for c in _read(Path(root)/MAIN_NOTEBOOK)['cells'] if c['cell_type']=='code'])

def study_configuration(run_id='deep-energy-pilot-001'):
    if not run_id or any(not(c.isalnum() or c in '-_') for c in run_id):raise ValueError('Safe nonempty run ID required')
    return {'schema':'fmm-jepa-notebook09-study/v1','run_id':run_id,'hard_limit_seconds':10800,
        'device':'cuda:0','dtype':'float32','image_size':128,'effective_batch':128,'microbatch':128,
        'seeds':[20261201,20261202],'variants':['vit','mlp','local','local_far'],'updates':350,
        'round_updates':50,'probe_epochs':20,'heldout_enabled':False,
        'deep_energy':{'depth':4,'summary_dim':32,'inner_steps':1,'near_strength':.25,'far_strength':.25,
            'step_size':.5,'coefficient_dim':32,'checkpoint':False,'edge_tile':1024},
        'analysis':{'role':'exploratory','n_pairs':2,'df':1,'confidence_level':.95,
            'contrasts':['local_minus_mlp','local_far_minus_local','local_far_minus_vit'],
            'endpoint':'Fixed epoch20 frozen-EMA-probe development top1 after350updates, percentage points',
            'interpretation':'Two-seed pilot for architecture screening; no superiority, equivalence or confirmatory significance claim. Shared pointwise backbone; active parameter counts differ.'},
        'primary_analysis':{'role':'exploratory_only','n_pairs':2,'df':1,'alpha':.05,
            'endpoint':'No confirmatory primary endpoint; report all registered exploratory contrasts and own-initialization gains.'},
        'secondary_analysis':'All gains, initialization-adjusted gains, time, memory and representation diagnostics are exploratory and not multiplicity-adjusted.',
        'budget_policy':'Fixed2freshpairs x4architectures x350updates; no reduction or extension.10percent timing allowance plus1800seconds evaluation and630seconds preflight/overhead. Deadline cutoff leaves INCOMPLETE.',
        'selection':'All4architectures,350updates,epoch20probes and2newseeds fixed before quality outcomes. No winner-based stopping or continuation.',
        'interpretation':'Pilot over training randomness conditional on reused development membership. Prior pilots and replication excluded from new outcomes; official test reserved.'}

def _status(artifact,stage,status,**details):
    result={'stage':stage,'status':status,'updated_unix':time.time(),**details}
    atomic_json(Path(artifact)/'study_status.json',result)
    print(stage+': '+status,flush=True)
    return result

def _validation(root):
    root=Path(root)
    operator=root/'artifacts/notebook_09_validation/operator_validation.json'
    report=_read(operator);current=_export_hash(root/'experiments/09_deep_energy/encoder.ipynb')
    if report.get('status')!='PASS' or report.get('encoder_export_sha256')!=current:
        raise ValueError('Notebook09 layered encoder must have matching PASS evidence')
    prior=root/'artifacts/notebook_07_validation/operator_validation.json'
    old=_read(prior)
    if old.get('status')!='PASS' or old.get('encoder_export_sha256')!=_export_hash(root/'experiments/07_energy/encoder.ipynb'):
        raise ValueError('Reused Notebook07 conditional energy operator certificate changed')
    pipeline=root/'artifacts/notebook_09_validation/runner_evaluation_validation.json'
    validation=_read(pipeline)
    if validation.get('status')!='PASS':raise ValueError('Notebook09 runner/evaluation validation must pass before launch')
    for name in ('runner','evaluation'):
        if validation.get(name+'_export_sha256')!=_export_hash(root/'experiments/09_deep_energy'/(name+'.ipynb')):
            raise ValueError('Pipeline validation does not match current '+name+' definitions')
    return {'operator':{'path':str(operator.relative_to(root)),'sha256':digest_file(operator),'encoder_export_sha256':current},
        'reused_operator':{'path':str(prior.relative_to(root)),'sha256':digest_file(prior)},
        'runner_evaluation':{'path':str(pipeline.relative_to(root)),'sha256':digest_file(pipeline)}}

def _check_binding(root,binding,config):
    reference='artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json'
    if binding.get('reference_pilot_plan')!={'path':reference,'sha256':digest_file(Path(root)/reference)}:
        raise ValueError('The pilot configuration source changed')
    if binding['configuration']!=_normalized(config):raise ValueError('Registered run configuration changed')
    if binding['source_hashes']!=source_hashes(root):raise ValueError('Registered source definitions changed; use a new run ID')
    if binding['consumer_notebook_sha256']!=consumer_hash(root):raise ValueError('Registered main notebook code changed')
    if binding['operator_validation']!=_validation(root):raise ValueError('Bound validation evidence changed')

def _clock(artifact,binding):
    result=_read(Path(artifact)/'budget_clock.json')
    if result!=binding['budget_clock'] or result['max_seconds']!=10800:raise ValueError('Original three-hour clock changed')
    return result

def _prepare(root,config):
    if _normalized(config)!=_normalized(study_configuration(config['run_id'])):
        raise ValueError('This pilot has a fixed registered configuration; no seed/update/architecture edits are permitted')
    root=Path(root).resolve();artifact=root/'artifacts/notebook_09'/config['run_id'];artifact.mkdir(parents=True,exist_ok=True)
    path=artifact/'study_binding.json'
    if path.exists():
        binding=_read(path);_check_binding(root,binding,config)
    else:
        if (artifact/'budget_clock.json').exists() or (artifact/'stages').exists():raise ValueError('Unbound existing experiment evidence; refusing restart')
        validation=_validation(root);now=time.time()
        clock={'started_unix':now,'deadline_unix':now+10800,'max_seconds':10800,'policy':'Original launch clock includes calibration, recovery verification, training and evaluation; never resets.'}
        binding={'schema':'notebook09-binding/v1','configuration':_normalized(config),'source_hashes':source_hashes(root),
                 'consumer_notebook_sha256':consumer_hash(root),'budget_clock':clock,'operator_validation':validation,
                 'reference_pilot_plan':{'path':'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json',
                    'sha256':digest_file(root/'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json')}}
        atomic_json(path,binding);atomic_json(artifact/'budget_clock.json',clock)
    return artifact,binding,_clock(artifact,binding)

def _choose_budget(config,calibration,deadline):
    if calibration['selected_microbatch']!=128 or calibration['selected_accumulation_steps']!=1:
        raise ValueError('Pilot must retain physical/effective batch128')
    timings=calibration['selected_model_step_seconds']
    if set(timings)!=set(config['variants']):raise ValueError('Every registered architecture needs calibration')
    if any(not isinstance(t,(int,float)) or not(0<t<float('inf')) for t in timings.values()):raise ValueError('Invalid calibration time')
    per_pair=sum(timings.values());remaining=deadline-time.time();count=len(config['seeds']);updates=config['updates']
    reserve=1800;preflight=450;overhead=180
    training=1.10*count*updates*per_pair
    result={'status':'PASS','seeds':count,'updates':updates,'architectures':len(config['variants']),
        'evaluation_reserve_seconds':reserve,'preflight_reserve_seconds':preflight,'overhead_reserve_seconds':overhead,
        'predicted_training_seconds':training,'predicted_total_remaining_seconds':training+reserve+preflight+overhead,
        'remaining_seconds':remaining,'calibration_round_seconds':per_pair,'fixed_schedule':True}
    if result['predicted_total_remaining_seconds']>remaining:
        raise TimeoutError('The full registered2-seed4-architecture350-update pilot does not fit the original3-hour clock; no smaller subset or schedule will be substituted: '+str(result))
    return result

def _plan(root,study,binding,bundle,base,calibration,budget):
    preparation=_load(root,'preparation');arms=[];seeds=study['seeds'];updates=study['updates']
    for index,seed in enumerate(seeds):
        for variant in study['variants']:
            cfg=preparation.arm_config(base,variant)
            cfg['deep09'].update(copy.deepcopy(study['deep_energy']));cfg['deep09']['variant']=variant
            cfg['training'].update(updates=updates,microbatch=calibration['selected_microbatch'],accumulation_steps=calibration['selected_accumulation_steps'])
            cfg['seeds'].update(model_init=seed,data_order=811101+1000*index,augmentation=822202+1000*index,mask=833303+1000*index,probe=4404)
            cfg['model']['feature_seed']=seed+1000
            cfg['evaluation'].update(batch_size=min(64,calibration['selected_microbatch']),diagnostic_steps=[0,updates])
            cfg['budget']['evaluation_reserve_seconds']=budget['evaluation_reserve_seconds']
            arms.append({'id':'s'+str(seed)+'-'+variant,'kind':'vit' if variant=='vit' else 'semantic','variant':variant,'seed':seed,'condition':variant,'config':cfg})
    return _normalized({'schema':'notebook09-stage-plan/v1','run_id':study['run_id'],'stage_id':'learning','stage_role':'learning',
         'stage':{'id':'learning','image_size':128,'pretrain_pool':'selected'},'updates':updates,'milestones':[updates],
         'round_updates':study['round_updates'],'effective_batch':128,'seeds':seeds,'variants':study['variants'],'arms':arms,
         'train_count':len(bundle['train_ids']),'probe_train_count':len(bundle['probe_train_ids']),
         'source_hashes':binding['source_hashes'],'consumer_notebook_sha256':binding['consumer_notebook_sha256'],
         'resume_notebook':MAIN_NOTEBOOK,'resume_cell':'launch-deep-energy','evaluation_reserve_seconds':budget['evaluation_reserve_seconds'],
         'study_configuration_hash':_hash(study),'numerical_backend':'deep-energy09-v1','heldout_status':'NOT_RUN',
         'analysis':study['analysis'],'primary_analysis':study['primary_analysis'],'secondary_analysis':study['secondary_analysis'],
         'excluded_pilot_seeds':[20261011,20261012]+list(range(20261101,20261111)),
         'selection':study['selection'],'capacity_policy':'Shared patch stem,allfourpointwiseMLPs,normsandpredictor; no dummy mixerparameters; active counts differ and must be reported.',
         'failure_policy':'Stop first numerical failure; never skip samples.','budget_selection':budget})

def run_study(root,run_id):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_09'/run_id
    binding=_read(artifact/'study_binding.json');study=binding['configuration'];_check_binding(root,binding,study)
    clock=_clock(artifact,binding);deadline=clock['deadline_unix']
    if (artifact/'study_status.json').exists() and _read(artifact/'study_status.json').get('status')=='COMPLETED':return _read(artifact/'study_status.json')
    if time.time()>=deadline:raise TimeoutError('Original three-hour deadline expired')
    preparation,runner,evaluation=(_load(root,n) for n in ('preparation','runner','evaluation'))
    with run_writer(artifact):
        try:
            folder=artifact/'stages/learning';folder.mkdir(parents=True,exist_ok=True)
            (folder/'predictions').mkdir(exist_ok=True)
            base=preparation.base_config(root)
            base['deep09'].update(copy.deepcopy(study['deep_energy']))
            _status(artifact,'verify_training_and_development_data','RUNNING')
            bundle=preparation.prepare_bundle(root,folder,base)
            _status(artifact,'actual_update_calibration','RUNNING')
            calibration=preparation.calibrate(root,folder,base,bundle,deadline)
            plan_path=folder/'plan.json'
            if plan_path.exists():
                plan=_read(plan_path)
                if plan['source_hashes']!=binding['source_hashes']:raise ValueError('Saved plan sources changed')
                if plan['study_configuration_hash']!=_hash(study):raise ValueError('Saved plan study changed')
            else:
                budget=_choose_budget(study,calibration,deadline)
                plan=_plan(root,study,binding,bundle,base,calibration,budget)
                preparation.immutable(plan_path,plan)
            _status(artifact,'exact_resume_preflight','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'])
            runner.prepare_stage(root,folder,plan,bundle)
            preflight=runner.verify_stage_resume(root,folder,plan,bundle,deadline)
            atomic_json(folder/'preflight_result.json',preflight)
            if preflight.get('status')!='PASS':raise RuntimeError('Exact-resume preflight did not pass')
            _status(artifact,'evaluation_integration_preflight','RUNNING')
            scoring=evaluation.verify_evaluation_smoke(root,folder,plan,bundle,deadline)
            atomic_json(folder/'evaluation_preflight_result.json',scoring)
            if scoring.get('status')!='PASS':raise RuntimeError('Evaluation integration preflight failed')
            _check_binding(root,binding,study)
            _status(artifact,'paired_training','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'])
            training=runner.train_stage(root,folder,plan,bundle,deadline)
            atomic_json(folder/'training_result.json',training)
            _status(artifact,'frozen_feature_evaluation','RUNNING',updates=plan['updates'],seeds=plan['seeds'])
            result=evaluation.evaluate_stage(root,folder,plan,bundle,training,deadline)
            atomic_json(artifact/'study_results.json',result)
            if result.get('status') not in ('PASS','COMPLETE','COMPLETED'):raise TimeoutError('Evaluation incomplete under original deadline')
            _check_binding(root,binding,study)
            return _status(artifact,'complete','COMPLETED',elapsed_seconds=time.time()-clock['started_unix'],updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'],heldout_status='NOT_RUN',results=str(folder/'stage_results.md'))
        except BaseException as error:
            _status(artifact,'interrupted','INCOMPLETE' if isinstance(error,TimeoutError) else 'FAILED',error=type(error).__name__+': '+str(error),cause=repr(error.__cause__) if error.__cause__ else None,elapsed_seconds=time.time()-clock['started_unix'])
            raise

def _exists(session):return subprocess.run(['tmux','has-session','-t','='+session],capture_output=True).returncode==0

def launch_study(root,config):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_09'/config['run_id']
    with run_writer(artifact/'launch_lock'):
        artifact,binding,clock=_prepare(root,config);session='fmm-deep-energy-'+config['run_id']
        if _exists(session):
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            return {'status':'EXISTING_SESSION','session':session,'deadline_unix':clock['deadline_unix'],'artifact':str(artifact)}
        prior=_read(artifact/'study_status.json') if (artifact/'study_status.json').exists() else {}
        if prior.get('status')=='COMPLETED':return prior
        if time.time()>=clock['deadline_unix']:raise TimeoutError('Original deadline expired; Run All cannot reset it')
        setup=next(c.source for c in nbformat.read(root/MAIN_NOTEBOOK,as_version=4).cells if c.get('id')=='bootstrap-deep-energy' and 'ahr-export' in c.metadata.get('tags',[]))
        code=setup+'\nWORKFLOW.run_study(ROOT, '+repr(config['run_id'])+')\n'
        worker=artifact/'worker.ipynb'
        if worker.exists():
            if [c.source for c in nbformat.read(worker,as_version=4).cells if c.cell_type=='code']!=[code]:raise ValueError('Detached worker source changed')
        else:
            nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Notebook09 detached worker\nOutputs: this run folder; full-state checkpoints: checkpoints/'+config['run_id']+'/. Resume with unchanged Notebook09 Run All before the original three-hour deadline.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}}),worker)
        command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(worker)]
        shell='exec '+shlex.join(command)+' > '+shlex.quote(str(artifact/'worker.log'))+' 2>&1'
        launch={'session':session,'artifact':str(artifact),'command':command,'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d','deadline_unix':clock['deadline_unix'],'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'binding_hash':_hash(binding)}
        atomic_json(artifact/'launch.json',launch);_status(artifact,'launching','STARTING')
        try:
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
        except BaseException as error:_status(artifact,'launch_failed','FAILED',error=repr(error));raise
        return {'status':'LAUNCHED',**launch}

def inspect_study(root,run_id):
    artifact=Path(root)/'artifacts/notebook_09'/run_id
    if not (artifact/'study_status.json').exists():return {'status':'NOT_LAUNCHED'}
    status=_read(artifact/'study_status.json');clock=_read(artifact/'budget_clock.json')
    result={**status,'session':'fmm-deep-energy-'+run_id,'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'remaining_minutes':max(0,(clock['deadline_unix']-time.time())/60)}
    rows=[]
    for p in sorted((Path(root)/'checkpoints'/run_id/'stages/learning').glob('arms/*/manifest.json')):
        m=_read(p);rows.append({'arm':p.parent.name,'successful_updates':m.get('successful_updates',m.get('latest_step'))})
    result['saved_training_progress']=rows
    return result


def readiness(root,config):
    """Read-only prelaunch check; creates no run folder, budget clock or tmux worker."""
    root=Path(root).resolve()
    if _normalized(config)!=_normalized(study_configuration(config['run_id'])):raise ValueError('Registered configuration changed')
    validation=_validation(root)
    encoder=_load(root,'encoder');preparation=_load(root,'preparation')
    base=preparation.base_config(root);rows=[]
    from fmm_jepa.food101_training import isolated_rng
    for variant in config['variants']:
        cfg=copy.deepcopy(base['model']);cfg['deep09']={**config['deep_energy'],'variant':variant}
        with isolated_rng(base['seeds']['model_init']):model=encoder.build_model(cfg,kind='vit' if variant=='vit' else 'semantic')
        counts=encoder.parameter_counts(model)
        total=counts['encoder_total'];inactive=counts['encoder_inactive_scaffold']
        rows.append({'architecture':variant,'depth':len(model.context.blocks),'encoder_parameters_total':total,
            'encoder_parameters_active':total-inactive,'inactive_parameters':inactive,
            'trainable_parameters_total':sum(p.numel() for p in model.parameters() if p.requires_grad)})
        del model
    artifact=root/'artifacts/notebook_09'/config['run_id']
    started=(artifact/'budget_clock.json').exists()
    current=_read(artifact/'study_status.json') if (artifact/'study_status.json').exists() else {}
    scientific='COMPLETE' if current.get('status')=='COMPLETED' else ('NOT_RUN' if not started else current.get('status','IN_PROGRESS'))
    return {'status':'EXISTING_STUDY' if started else 'READY_TO_LAUNCH','production_run_started':started,'expected_runtime':'Approximately2–3hours; actual feasibility checked at launch; hard cap3hours.',
        'architecture_summary':rows,'validation':validation,'run_id':config['run_id'],'planned_training_runs':8,'planned_probe_evaluations':16,
        'checkpoint_location':str(root/'checkpoints'/config['run_id']), 'scientific_results':scientific}
